# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/satkynovilgiz/flyrank-ml-content-refresh/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.620


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.800
Precision@50:  hand rule 0.620   vs   tree 0.720


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
# Your turn: client-holdout check (plus max_depth 2 vs 3 vs 4). Train on some clients, score on clients the tree never saw.
# Same idea as scripts/03_train_model.py (hold out ~20% of clients), done with GroupShuffleSplit.
# Features are the same pre-decision list as above: no trend_direction / trend_pct (label source),
# no content_id / client_id (IDs are for grouping only).
from sklearn.model_selection import GroupShuffleSplit

assert not {"trend_direction", "trend_pct", "content_id", "client_id"} & set(features)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
groups = df["client_id"].to_numpy()
assert not set(groups[train_idx]) & set(groups[test_idx]), "a client is in both train and test"

print(f"Train: {len(set(groups[train_idx]))} clients, {len(train_idx):,} pages | "
      f"Test: {len(set(groups[test_idx]))} clients, {len(test_idx):,} pages")
print(f"Declining rate  train {y[train_idx].mean():.3f}  |  test {y[test_idx].mean():.3f}\n")

def expected_precision_at_k(scores, labels, k):
    """Precision@K if tied scores were broken at random (tie-robust)."""
    scores, labels = np.asarray(scores), np.asarray(labels)
    hits, left = 0.0, k
    for s in np.sort(np.unique(scores))[::-1]:
        block = labels[scores == s]
        take = min(left, len(block))
        hits += take * block.mean()
        left -= take
        if left == 0:
            break
    return hits / k

hand = df["hand_rule_score"].to_numpy()
# How many pages does the hand rule actually flag (score > 0)? Everything else is tied at 0.
for name, sc, lab in (("all pages", hand, y), ("test clients", hand[test_idx], y[test_idx])):
    flagged = sc > 0
    rate = f"{lab[flagged].mean():.3f}" if flagged.any() else "n/a"
    print(f"hand rule flags {int(flagged.sum()):,} of {len(sc):,} {name} "
          f"(declining rate among flagged: {rate}); the rest are tied at score 0")
rows = []
rows.append({"ranker": "hand rule", "metric": "P@50 tie-robust",
             "in-sample (all pages)": expected_precision_at_k(hand, y, 50),
             "holdout (test clients)": expected_precision_at_k(hand[test_idx], y[test_idx], 50)})
for k in (20, 50):
    rows.append({"ranker": "hand rule", "metric": f"Precision@{k}",
                 "in-sample (all pages)": precision_at_k(hand, y, k),
                 "holdout (test clients)": precision_at_k(hand[test_idx], y[test_idx], k)})
for depth in (2, 3, 4):
    t_all = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42).fit(X, y)
    t_tr  = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42).fit(
                X.iloc[train_idx], y[train_idx])
    s_all = t_all.predict_proba(X)[:, 1]
    s_te  = t_tr.predict_proba(X.iloc[test_idx])[:, 1]
    for k in (20, 50):
        rows.append({"ranker": f"tree depth {depth}", "metric": f"Precision@{k}",
                     "in-sample (all pages)": precision_at_k(s_all, y, k),
                     "holdout (test clients)": precision_at_k(s_te, y[test_idx], k)})
    rows.append({"ranker": f"tree depth {depth}", "metric": "P@50 tie-robust",
                 "in-sample (all pages)": expected_precision_at_k(s_all, y, 50),
                 "holdout (test clients)": expected_precision_at_k(s_te, y[test_idx], 50)})
    # How many test pages share the single highest tree score? A big tie block makes P@K fragile.
    top_tie = int((s_te == s_te.max()).sum())
    print(f"depth {depth}: {len(np.unique(s_te))} distinct scores on test; "
          f"{top_tie:,} test pages tied at the top score")

holdout_table = pd.DataFrame(rows).pivot(index="ranker", columns="metric",
                                         values=["in-sample (all pages)", "holdout (test clients)"])
print()
print(holdout_table.round(3).to_string())

print("\nDepth-3 tree trained on train clients only (readable?):")
t3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(
         X.iloc[train_idx], y[train_idx])
print(export_text(t3, feature_names=features))


Train: 25 clients, 23,837 pages | Test: 7 clients, 6,163 pages
Declining rate  train 0.550  |  test 0.511

hand rule flags 17 of 30,000 all pages (declining rate among flagged: 0.941); the rest are tied at score 0
hand rule flags 0 of 6,163 test clients (declining rate among flagged: n/a); the rest are tied at score 0
depth 2: 4 distinct scores on test; 2,963 test pages tied at the top score
depth 3: 8 distinct scores on test; 2,209 test pages tied at the top score


depth 4: 15 distinct scores on test; 1,319 test pages tied at the top score

             in-sample (all pages)                           holdout (test clients)                          
metric             P@50 tie-robust Precision@20 Precision@50        P@50 tie-robust Precision@20 Precision@50
ranker                                                                                                       
hand rule                    0.678         0.90         0.62                  0.511         0.65         0.44
tree depth 2                 0.651         0.80         0.72                  0.579         0.55         0.60
tree depth 3                 0.686         0.55         0.68                  0.589         0.65         0.60
tree depth 4                 0.716         0.70         0.66                  0.613         0.70         0.64

Depth-3 tree trained on train clients only (readable?):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50


### My experiment: does the in-sample gap hold on unseen clients?

**Question.** In-sample, the depth-2 tree beat my hand rule at Precision@50 (0.72 vs 0.62), while the hand rule won at Precision@20 (0.90 vs 0.80). Does that still hold when I score on clients the tree never saw?

**What I did.** I split the pages with `GroupShuffleSplit` grouped by `client_id` (test_size 0.2, random_state 42), the same idea as `scripts/03_train_model.py`. That gave 25 train clients (23,837 pages) and 7 test clients (6,163 pages), with no client on both sides. I fit the same readable tree, with the same 6 features, on the train clients only, at max_depth 2, 3 and 4. I left out `trend_direction`, `trend_pct` (the label source) and the IDs. Then I scored the hand rule and each tree on the test clients with Precision@20 and Precision@50. Both trees and the rule have large blocks of tied scores, so I also added a tie-robust Precision@50: what you'd expect if ties were broken at random.

**Observed.**

| ranker | P@50 in-sample | P@50 holdout | P@50 tie-robust holdout | P@20 holdout |
|---|---|---|---|---|
| hand rule | 0.62 | 0.44 | 0.511 | 0.65 |
| tree depth 2 | 0.72 | 0.60 | 0.579 | 0.55 |
| tree depth 3 | 0.68 | 0.60 | 0.589 | 0.65 |
| tree depth 4 | 0.66 | 0.64 | 0.613 | 0.70 |

- Test-client base rate (share of pages declining): 0.511.
- **The big surprise: the hand rule barely fires.** It flags only 17 of 30,000 pages (16 of those 17 are declining, 0.941), because hardly any pages have `days_since_last_update >= 180`. On the 7 test clients it flags **0** pages. All 6,163 holdout pages tie at score 0, so the rule's holdout 0.44 is just file order, and its tie-robust 0.511 is exactly the base rate. Its strong in-sample Precision@20 (0.90) rests on those 17 pages.
- The tree's edge does hold on unseen clients, though smaller: tie-robust P@50 of 0.58–0.61 against the 0.511 base rate, roughly 3–5 more correct pages out of 50.
- Depth 3 vs 4: depth 4 came out a little higher on the holdout (tie-robust 0.613 vs 0.589; P@50 0.64 vs 0.60). That gap is about 1–2 pages out of 50, so I treat it as noise-level. The depth-3 tree printed above is still easy to read (impressions, then content age, then ctr / position). Depth 4 has up to 16 leaves and is harder to read.

**Caveat.** This is one split with only 7 test clients, and at K = 50 each page moves the score by 0.02. The tree scores are heavily tied (1,319–2,963 test pages share the top score), so plain Precision@K partly depends on row order. This is one anonymized slice. It's directional, decision-support evidence that the tree ranks better than chance on new clients, not proof that it generalises.

**Next steps.**
1. Repeat the client split many times (e.g. 20 GroupShuffleSplit seeds or GroupKFold) and report the mean and spread instead of one number.
2. Fix the hand rule's coverage: the 180-day "stale" cutoff almost never triggers here. Try a per-client percentile or a lower cutoff so the rule says something about every client.
3. Compare against the pipeline's random forest on its own client holdout (Precision@50 = 0.68 in my run).


In [8]:
# Your experiment here


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.